# Spatial Testing

## Notebook Preferences

In [ ]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format="retina"

In [ ]:
import nbl
import squidpy as sq
import lamindb as ln
import spatialdata_plot  # noqa: F401
import spatialdata as sd
import buckaroo
import narwhals as nw
from narwhals.typing import IntoFrameT
import anndata as ad
import numpy as np
import matplotlib.pyplot as plt
import anndata_tools  # noqa: F401
from anndata_tools import ObsCol

In [ ]:
ln.settings.sync_git_repo = "https://github.com/karadavis-lab/nbl.git"
ln.track("DQSGNeBh8MsV0000")

### Load Data

In [ ]:
nbl_sdata = sd.read_zarr(ln.Artifact.get(description="NBL Tissue Samples", is_latest=True).path)

In [ ]:
nbl_adata = nbl_sdata.tables["arcsinh_shift_0_scale_150"]

In [ ]:
@nw.narwhalify
def select_classification(obs_df: IntoFrameT, classification: str) -> IntoFrameT:
    """Selects the classification column from the observation dataframe."""
    return obs_df.filter(nw.col("Classification").is_in([classification]))


def subset_sd_by_classification(adata: ad.AnnData, classification: str):
    """Subsets the SpatialData object by the classification column."""
    subset_obs_index = select_classification(obs_df=adata.obs, classification=classification).index
    return adata[subset_obs_index, :].copy()

In [ ]:
nbl_adata.sel.filter(ObsCol("Classification") == "Diagnosis", copy=True)

In [ ]:
buckaroo.disable()

In [ ]:
nbl_sdata.tables["diagnosis"] = nbl_sdata.update_annotated_regions_metadata(
    table=subset_sd_by_classification(nbl_adata, classification="Diagnosis")
)
nbl_adata = nbl_sdata.tables["diagnosis"]

In [ ]:
sq.gr.spatial_neighbors(nbl_adata, spatial_key="spatial", library_key="region", n_neighs=10, transform=None)

In [ ]:
nbl_adata

In [ ]:
rng = np.random.default_rng(seed=0)

In [ ]:
sq.gr.nhood_enrichment(nbl_adata, library_key="region", cluster_key="pixie_cluster", connectivity_key="spatial")

Compute neighborhood enrichment by permutation test.



In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(10, 10), dpi=300, layout="constrained")

sq.pl.nhood_enrichment(adata=nbl_adata, cluster_key="pixie_cluster", annotate=True, ax=ax)
fig.savefig(fname="./nhood_enrichment.pdf", dpi=300, bbox_inches="tight")

Co-occurence probability of clusters

In [ ]:
sq.gr.co_occurrence(adata=nbl_adata, cluster_key="pixie_cluster", spatial_key="spatial", interval=50, n_jobs=10)

In [ ]:
for cluster in nbl_adata.obs["pixie_cluster"].cat.categories:
    sq.pl.co_occurrence(
        adata=nbl_adata,
        cluster_key="pixie_cluster",
        clusters=[cluster],
        legend_kwargs={"fontsize": "x-small"},
        dpi=600,
        save=f"./{cluster}_co_occurrence.pdf",
        figsize=(10, 10),
    )

Centrality scores per cluster or cell type

In [ ]:
sq.gr.centrality_scores(nbl_adata, cluster_key="pixie_cluster", score=None, n_jobs=8)

In [ ]:
for score_metric in ["average_clustering", "degree_centrality", "closeness_centrality"]:
    sq.pl.centrality_scores(
        nbl_adata,
        cluster_key="pixie_cluster",
        score=score_metric,
        figsize=(12, 5),
        dpi=600,
        save=f"./{score_metric}_centrality_scores.pdf",
    )

In [ ]:
import spaco

In [ ]:
color_mapping = spaco.colorize(
    cell_coordinates=nbl_adata.obsm["spatial"],
    cell_labels=nbl_adata.obs["pixie_cluster"],
    colorblind_type="none",
    radius=0.05,
    n_neighbors=30,
)

In [ ]:
color_mapping = {k: color_mapping[k] for k in nbl_adata.obs["pixie_cluster"].cat.categories}
palette_spaco = list(color_mapping.values())

Interaction Matrix

In [ ]:
sq.gr.interaction_matrix(adata=nbl_adata, cluster_key="pixie_cluster", normalized=True)

In [ ]:
sq.pl.interaction_matrix(
    nbl_adata,
    cluster_key="pixie_cluster",
    title="Interaction matrix",
    annotate=True,
    figsize=(10, 10),
    dpi=600,
    save="./interaction_matrix.pdf",
)

Spatial Autocorrelation

In [ ]:
nbl.ln.featuresets.MarkerSet.NEUROBLASTOMA.to_list()

In [ ]:
sq.gr.spatial_autocorr(nbl_adata, genes=nbl.ln.featuresets.MarkerSet.NEUROBLASTOMA.to_list(), n_perms=10)

In [ ]:
sq.pl.spatial

Sepal

In [ ]:
sq.gr.sepal(nbl_adata, genes=nbl.ln.featuresets.MarkerSet.NEUROBLASTOMA.to_list(), max_neighs=4)